In [8]:
!nvidia-smi

Mon Feb  2 16:53:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             30W /   70W |    1336MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Data Loading & Filtering

In [11]:
# Load Labels
csv_path = '/content/drive/MyDrive/Colab Notebooks/labels_synthetic_calibrated_with_path.csv'

# Check if file exists (Colab path might be different, e.g., /content/labels...)
if not os.path.exists(csv_path):
    print(f"Warning: {csv_path} not found. Please update 'csv_path' to your location.")
    # Example for Colab upload:
    # csv_path = '/content/labels_synthetic_calibrated.csv'

df = pd.read_csv(csv_path)

# Filter for Slit-Lamp images
# The 'source' column should distinguish between 'mobile' and 'slit_lamp'
print("Available sources:", df['source'].unique())

target_source = 'slit_lamp'
if target_source not in df['source'].values:
    # Fallback if named 'slitlamp'
    if 'slitlamp' in df['source'].values:
        target_source = 'slitlamp'

slitlamp_df = df[df["source"] == target_source].copy()
print(f"Filtered {len(slitlamp_df)} slit-lamp images.")

# Ensure relative path column exists
if 'relative_path' not in slitlamp_df.columns:
    # Fallback: construct path if needed, or error out
    print("Error: 'relative_path' column missing. Please ensure CSV is updated.")
else:
    print("Path column found.")

Available sources: ['mobile' 'slit_lamp']
Filtered 7026 slit-lamp images.
Path column found.


## 2. Dataset Definition

In [12]:
class CataractDataset(Dataset):
    def __init__(self, dataframe, root_dir=None, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir if root_dir else ''
        self.transform = transform

        # Targets: NO, NC, CO, PSC
        self.targets = ['NO_pseudo', 'NC_pseudo', 'CO_pseudo', 'PSC_pseudo']

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Construct image path
        # If running in Colab, root_dir might be '/content/'
        img_rel_path = row['relative_path'].replace('\\', '/')
        img_path = os.path.join(self.root_dir, img_rel_path)

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a dummy tensor or handle error (here we fail noisy)
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        # Get labels (continuous or discrete? The concept model predicts continuous 0-5)
        # If using pseudo-labels (bins), we can convert to float
        labels = row[self.targets].values.astype(float)
        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels

## 3. Train/Val Split & DataLoader

In [13]:
# Split into Train and Validation
train_df, val_df = train_test_split(slitlamp_df, test_size=0.2, random_state=42)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Datasets
# root_dir should be adjusted if data is inside a subfolder
data_root = '/content/drive/MyDrive/Colab Notebooks'
train_dataset = CataractDataset(train_df, root_dir=data_root, transform=train_transform)
val_dataset = CataractDataset(val_df, root_dir=data_root, transform=val_transform)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

Train size: 5620, Val size: 1406


## 4. Model Definition

In [14]:
class ConceptPredictor(nn.Module):
    def __init__(self, backbone_name='resnet18', pretrained=True):
        super(ConceptPredictor, self).__init__()

        # Load backbone
        if backbone_name == 'resnet18':
            self.backbone = models.resnet18(pretrained=pretrained)
            in_features = self.backbone.fc.in_features
            # Remove the classification head
            self.backbone.fc = nn.Identity()
        elif backbone_name == 'efficientnet_b0':
            self.backbone = models.efficientnet_b0(pretrained=pretrained)
            in_features = self.backbone.classifier[1].in_features
            # Remove classification head
            self.backbone.classifier = nn.Identity()
        else:
            raise ValueError(f"Backbone {backbone_name} not supported.")

        # Concept Head: 4 neurons for NO, NC, CO, PSC
        self.concept_head = nn.Linear(in_features, 4)

    def forward(self, x):
        features = self.backbone(x)
        raw_output = self.concept_head(features)

        # Clamp output to valid range [0, 5]
        clamped_output = torch.clamp(raw_output, 0, 5)

        return clamped_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ConceptPredictor(backbone_name='resnet18', pretrained=True).to(device)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## 5. Training Loop

In [15]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_loss:.4f} Val Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'concept_slitlamp.pt')
        print("  Model saved!")

print("Training Complete.")

save_dir = '/content/drive/MyDrive/Colab Notebooks/Saved Models'
model_filename = 'concept_slitlamp_final.hdf5'
save_path = os.path.join(save_dir, model_filename)

os.makedirs(save_dir, exist_ok=True)

# Save the model's state dictionary
torch.save(model.state_dict(), save_path)

print(f"Model saved to: {save_path}")

Epoch [1/10] Train Loss: 0.4023 Val Loss: 0.1442
  Model saved!
Epoch [2/10] Train Loss: 0.1434 Val Loss: 0.1414
  Model saved!
Epoch [3/10] Train Loss: 0.1195 Val Loss: 0.1024
  Model saved!
Epoch [4/10] Train Loss: 0.1074 Val Loss: 0.1013
  Model saved!
Epoch [5/10] Train Loss: 0.0914 Val Loss: 0.0937
  Model saved!
Epoch [6/10] Train Loss: 0.0838 Val Loss: 0.0925
  Model saved!
Epoch [7/10] Train Loss: 0.0741 Val Loss: 0.0789
  Model saved!
Epoch [8/10] Train Loss: 0.0649 Val Loss: 0.0856
Epoch [9/10] Train Loss: 0.0623 Val Loss: 0.0881
Epoch [10/10] Train Loss: 0.0536 Val Loss: 0.0842
Training Complete.
